In [10]:
import sys
import toml
from openai import OpenAI
from pathlib import Path
import json
from typing import Dict, Any, List
import docx
import tiktoken

from api_utils import load_api_params
from doc_utils import load_document


In [11]:
SECRETS_PATH = ".secrets.toml" 

In [12]:
# Load API parameters and initialize client
API_CALL_PARAMS = load_api_params(SECRETS_PATH)
client = OpenAI(
    base_url=API_CALL_PARAMS['API_URL'],
    api_key=API_CALL_PARAMS['API_KEY']
)

In [13]:
DOC_PATH = "mock_docs/interview_transcript_01.docx"  
document = load_document(DOC_PATH)
print(document)

Interview Transcript: Art Tutor Software Focus Group
Date: January 25, 2025
Participant: Sarah Chen, Digital Artist
Interviewer: Marcus Reynolds

MR: Thanks for joining us today. How long have you been using Art Tutor?
SC: About three months now. I have... opinions.
MR: Please share them.
SC: Well, the AI brush prediction feature is brilliant when it works. It's like having a master artist guiding your hand. But holy cow, does it get temperamental! Sometimes it just decides to give me completely wrong suggestions. Like, I'm clearly working on a portrait, and it suggests brushstrokes for landscape textures. Super frustrating.
MR: How often does this happen?
SC: Maybe 20% of the time? Look, I absolutely love the color harmony suggestions though. That's a game-changer. It's saved me countless hours of tweaking palettes. The way it analyzes art history to suggest historically accurate color combinations? *chef's kiss*
MR: Tell me about the tutorial system.
SC: [Laughs sarcastically] Oh, th

In [14]:
# Model Parameters (change if needed)
TEMPERATURE = 0.3
STREAM = False

def generate_completion(model: str, messages: List[Dict[str, str]]) -> str:
    response = client.chat.completions.create(
        model=model, 
        messages=messages,
        temperature=TEMPERATURE,
        stream=STREAM
    )
    return response.choices[0].message.content

In [15]:
CODING_PROMPT_01 = """You are a skilled qualitative researcher conducting thematic analysis. Analyze the provided data following these steps:

1. Initial Coding:
- Extract significant quotes/passages
- Assign descriptive codes
- Note patterns and relationships

2. Theme Development:
- Group related codes into potential themes
- Explain why these groupings are meaningful
- Identify hierarchy (main themes vs. subthemes)

3. For each identified theme:
- Provide clear definition
- Include 2-3 supporting quotes/examples
- Explain its relationship to research questions
- Note any contradictions or tensions

4. Quality Assessment:
- Address potential alternative interpretations
- Discuss any limitations in the analysis
- Note areas requiring additional data/clarification

Format your response as:

Overview:
[Brief context and objectives]

Themes:
1. [Theme Name]
- Definition:
- Supporting Evidence:
- Relationship to Research Questions:
- Tensions/Contradictions:

[Repeat for each theme]

Analysis Quality:
[Discussion of limitations and alternatives]

Remember to:
- Support all claims with direct evidence from the data
- Maintain analytical depth while being concise
- Consider contradictory evidence
- Focus on patterns relevant to research objectives"""

In [16]:
#MODEL = 'meta-llama/llama-3.2-1b-instruct'
MODEL = 'google/gemini-2.0-flash-thinking-exp-1219:free'

messages = [
    {"role": "system", "content": "You are a qualitative researcher analyzing textual data."},
    {"role": "user", "content":f"""{CODING_PROMPT_01} {document}"""}
]
try:
    list_first_pass = generate_completion(MODEL, messages)
except Exception as e:
    raise Exception(f"Error generating completion: {e}")

In [17]:
total_text = messages[0]['content'] + "\n" + messages[1]['content']
encoder = tiktoken.encoding_for_model("gpt-3.5-turbo")
tokens = encoder.encode(total_text)
num_tokens = len(tokens)

metadata = {
    "model": MODEL,
    "doc_path": DOC_PATH,
    "estimated_tokens": num_tokens
}

In [18]:
print(json.dumps(metadata) + "\n" + list_first_pass)

{"model": "meta-llama/llama-3.2-1b-instruct", "doc_path": "mock_docs/interview_transcript_01.docx", "estimated_tokens": 938}
Okay, I'm ready to analyze the interview transcript and provide a thematic analysis as requested.

**Overview:**
This analysis examines a focus group interview transcript with Sarah Chen, a digital artist using Art Tutor software. The objective is to understand Sarah's user experience, identify key strengths and weaknesses of the software, and highlight areas for potential improvement from a user perspective. The analysis will focus on identifying recurring themes related to software features, usability, and overall value.

**Themes:**

**1. AI Feature Experience: A Double-Edged Sword**
- Definition: This theme captures Sarah's mixed experiences with the AI-powered features of Art Tutor, specifically the AI brush prediction and color harmony suggestions. It highlights both the potential and the current limitations of these features.
- Supporting Evidence:
    - "